<a href="https://colab.research.google.com/github/AIVIETNAM-AIO-Tuan/AIO-Conquer/blob/MinhKhoa/Model/test_obesity_lassovsboruta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Boruta -q
!pip install statsmodels -q

import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from boruta import BorutaPy
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV

In [ ]:
drive.mount('/content/drive')
FILE_PATH = '/content/drive/MyDrive/conquer1/AIO-Conquer-Data/Data/dataset/obesity_encoded.csv'
try:
    df = pd.read_csv(FILE_PATH)
    print(f"Đã tải dữ liệu thành công! Kích thước: {df.shape}")
    df.info()
    df = df.copy()
except Exception as e:
    print(f"Lỗi: Không tìm thấy file hoặc đường dẫn sai. Chi tiết: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã tải dữ liệu thành công! Kích thước: (2087, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2087 entries, 0 to 2086
Data columns (total 33 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Age                                 2087 non-null   float64
 1   Height                              2087 non-null   float64
 2   Weight                              2087 non-null   float64
 3   FCVC                                2087 non-null   float64
 4   NCP                                 2087 non-null   float64
 5   CH2O                                2087 non-null   float64
 6   FAF                                 2087 non-null   float64
 7   TUE                                 2087 non-null   float64
 8   NObeyesdad                          2087 non-null   int64  

In [ ]:
target_colum = 'Weight'
bool_col = ['Gender_Male','family_history_with_overweight_yes','FAVC_yes','SMOKE_yes','SCC_yes']
X = df.drop(columns=[target_colum,'BMI','Gender_Male','family_history_with_overweight_yes','FAVC_yes','SMOKE_yes','SCC_yes'], errors='ignore')
y = df[target_colum]

# 2. Chia tập Train/Test trước để tránh data leakage khi tính trung bình giá
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
lr_model = LinearRegression()

# 2. Huấn luyện mô hình trên tập dữ liệu đã encode
lr_model.fit(X_train, y_train)

# 3. Tiến hành dự đoán cân nặng
preds = lr_model.predict(X_test)

# 4. Đánh giá mô hình bằng R2-score, RMSE, MAE
r2_score_all = round(r2_score(y_test, preds), 3)
rmse_all = round(np.sqrt(mean_squared_error(y_test, preds)), 3)
mae_all = round(mean_absolute_error(y_test, preds), 3)

print(f"Chỉ số R2-score khi sử dụng Linear Regression: {r2_score_all}")
print(f"Chỉ số RMSE khi sử dụng Linear Regression: {rmse_all}")
print(f"Chỉ số MAE khi sử dụng Linear Regression: {mae_all}")

Chỉ số R2-score khi sử dụng Linear Regression: 0.96
Chỉ số RMSE khi sử dụng Linear Regression: 5.327
Chỉ số MAE khi sử dụng Linear Regression: 4.119


# **1.Boruta**

In [ ]:
# --- 1. Cấu hình và chạy Boruta ---
# BorutaPy yêu cầu mảng NumPy (.values) để tránh lỗi index
X_train_np = X_train.values
y_train_np = y_train.values

# Giữ nguyên Random Forest làm mô hình nền để Boruta tính toán feature_importances_
rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
boruta_selector = BorutaPy(estimator=rf, n_estimators='auto', verbose=0, random_state=42, max_iter=100)
boruta_selector.fit(X_train_np, y_train_np)

# --- 2. Lọc và huấn luyện lại mô hình ---
# Lọc lấy các tính năng quan trọng được Boruta xác nhận
X_train_filtered = boruta_selector.transform(X_train_np)
X_test_filtered = boruta_selector.transform(X_test.values)

#Khởi tạo và huấn luyện mô hình Linear Regression trên tập dữ liệu đã lọc
lr_final = LinearRegression()
lr_final.fit(X_train_filtered, y_train_np)

# --- 3. Dự đoán và đánh giá điểm R2 ---
# Sử dụng mô hình Linear Regression vừa train để dự đoán
preds_boruta = lr_final.predict(X_test_filtered)
r2_score_boruta = round(r2_score(y_test.values, preds_boruta), 3)
rsme_boruta = round(np.sqrt(mean_squared_error(y_test.values, preds_boruta)), 3)
mae_boruta = round(mean_absolute_error(y_test.values, preds_boruta), 3)

# --- 4. Trích xuất danh sách tính năng được chọn ---
feature_names = X_train.columns
confirmed_features = feature_names[boruta_selector.support_].tolist()

# --- 5. Xuất kết quả báo cáo ---
print(f"{'KẾT QUẢ TÁC ĐỘNG CỦA BORUTA LÊN MÔ HÌNH':^60}")
print(f"• Chỉ số R2-score (Linear Regression) : {r2_score_boruta}")
print(f"• Chỉ số RMSE (Linear Regression)    : {rsme_boruta}")
print(f"• Chỉ số MAE (Linear Regression)     : {mae_boruta}")
print(f"• Số lượng tính năng được giữ lại   : {len(confirmed_features)}/{len(feature_names)}")
print(f"• Các tính năng được giữ lại:\n  {confirmed_features}")
print("="*60)

          KẾT QUẢ TÁC ĐỘNG CỦA BORUTA LÊN MÔ HÌNH           
• Chỉ số R2-score (Linear Regression) : 0.96
• Chỉ số RMSE (Linear Regression)    : 5.335
• Chỉ số MAE (Linear Regression)     : 4.095
• Số lượng tính năng được giữ lại   : 14/26
• Các tính năng được giữ lại:
  ['Age', 'Height', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'NObeyesdad', 'Gender_Female', 'FAVC_no', 'CALC_Sometimes', 'CALC_no', 'MTRANS_Automobile', 'MTRANS_Public_Transportation']


# **2.LASSO**

In [ ]:
# --- 1. Chuẩn hóa dữ liệu (Feature Scaling) ---
# Bước này bắt buộc phải có vì Lasso rất nhạy cảm với scale của dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 2. Khởi tạo và chạy LassoCV để SÀNG LỌC TÍNH NĂNG ---
lasso = LassoCV(alphas=np.logspace(-2,-1, 100), cv=5, random_state=42, n_jobs=-1)
lasso.fit(X_train_scaled, y_train)

# Trích xuất danh sách tên các tính năng được Lasso giữ lại (Coef != 0)
feature_names = X_train.columns
kept_features = feature_names[lasso.coef_ != 0].tolist()
dropped_features = feature_names[lasso.coef_ == 0].tolist()

# --- 3. Lọc lại dữ liệu theo các tính năng được chọn ---
# Lấy vị trí index của các tính năng được giữ lại để lọc trên mảng numpy đã scaled
kept_features_idx = [X_train.columns.get_loc(col) for col in kept_features]
X_train_filtered = X_train_scaled[:, kept_features_idx]
X_test_filtered = X_test_scaled[:, kept_features_idx]

# --- 4. Huấn luyện mô hình LINEAR REGRESSION trên các tính năng đã lọc ---
lr_final = LinearRegression(n_jobs=-1)
lr_final.fit(X_train_filtered, y_train)

# --- 5. Dự đoán và tính toán các chỉ số đánh giá ---
preds_lr = lr_final.predict(X_test_filtered)
r2_score_lr = round(r2_score(y_test, preds_lr), 3)
rmse_lr = round(np.sqrt(mean_squared_error(y_test, preds_lr)), 3)
mae_lr = round(mean_absolute_error(y_test, preds_lr), 3)

# --- 6. Xuất kết quả báo cáo ---
print(f"{'KẾT QUẢ TÁC ĐỘNG CỦA LASSO LÊN MÔ HÌNH':^60}")
print(f"• Alpha tối ưu của Lasso             : {round(lasso.alpha_, 5)}")
print(f"• Chỉ số R2-score (Linear Regression) : {r2_score_lr}")
print(f"• Chỉ số RMSE (Linear Regression)    : {rmse_lr}")
print(f"• Chỉ số MAE (Linear Regression)     : {mae_lr}")
print(f"• Số lượng tính năng bị loại bỏ      : {len(dropped_features)}/{len(feature_names)}")
print(f"• Số lượng tính năng được giữ lại    : {len(kept_features)}/{len(feature_names)}")
print("-"*60)
if dropped_features:
    print(f"• Các tính năng bị loại bỏ (Lasso Coef = 0):\n  {dropped_features}\n")
print(f"• Các tính năng được chọn để chạy Linear Regression:\n  {kept_features}")
print("="*60)

           KẾT QUẢ TÁC ĐỘNG CỦA LASSO LÊN MÔ HÌNH           
• Alpha tối ưu của Lasso             : 0.03126
• Chỉ số R2-score (Linear Regression) : 0.96
• Chỉ số RMSE (Linear Regression)    : 5.327
• Chỉ số MAE (Linear Regression)     : 4.119
• Số lượng tính năng bị loại bỏ      : 3/26
• Số lượng tính năng được giữ lại    : 23/26
------------------------------------------------------------
• Các tính năng bị loại bỏ (Lasso Coef = 0):
  ['CAEC_Always', 'CALC_Frequently', 'MTRANS_Walking']

• Các tính năng được chọn để chạy Linear Regression:
  ['Age', 'Height', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'NObeyesdad', 'Gender_Female', 'family_history_with_overweight_no', 'FAVC_no', 'CAEC_Frequently', 'CAEC_Sometimes', 'CAEC_no', 'SMOKE_no', 'SCC_no', 'CALC_Always', 'CALC_Sometimes', 'CALC_no', 'MTRANS_Automobile', 'MTRANS_Bike', 'MTRANS_Motorbike', 'MTRANS_Public_Transportation']


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoLarsIC, LinearRegression  # Thay LassoCV bằng LassoLarsIC
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# --- 1. Chuẩn hóa dữ liệu (Feature Scaling) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 2. Khởi tạo và chạy LassoLarsIC để SÀNG LỌC TÍNH NĂNG bằng BIC ---
# criterion='bic' giúp cấu hình mô hình phạt theo tiêu chí BIC
# (Bạn cũng có thể đổi thành 'aic' nếu muốn phạt nhẹ hơn một chút)
lasso_bic = LassoLarsIC(criterion='bic', normalize=False)
lasso_bic.fit(X_train_scaled, y_train)

# Trích xuất danh sách tên các tính năng được Lasso giữ lại (Coef != 0)
feature_names = X_train.columns
kept_features = feature_names[lasso_bic.coef_ != 0].tolist()
dropped_features = feature_names[lasso_bic.coef_ == 0].tolist()

# --- 3. Lọc lại dữ liệu theo các tính năng được chọn ---
kept_features_idx = [X_train.columns.get_loc(col) for col in kept_features]
X_train_filtered = X_train_scaled[:, kept_features_idx]
X_test_filtered = X_test_scaled[:, kept_features_idx]

# --- 4. Huấn luyện mô hình LINEAR REGRESSION trên các tính năng đã lọc ---
# Thêm kiểm tra nếu kept_features trống (trường hợp BIC loại bỏ sạch 100% biến)
if len(kept_features) == 0:
    print("CẢNH BÁO: BIC quá nghiêm ngặt và đã loại bỏ toàn bộ các biến!")
else:
    lr_final = LinearRegression(n_jobs=-1)
    lr_final.fit(X_train_filtered, y_train)

    # --- 5. Dự đoán và tính toán các chỉ số đánh giá ---
    preds_lr = lr_final.predict(X_test_filtered)
    r2_score_lr = round(r2_score(y_test, preds_lr), 3)
    rmse_lr = round(np.sqrt(mean_squared_error(y_test, preds_lr)), 3)
    mae_lr = round(mean_absolute_error(y_test, preds_lr), 3)

    # --- 6. Xuất kết quả báo cáo ---
    print(f"{'KẾT QUẢ TÁC ĐỘNG CỦA LASSO (BIC) LÊN MÔ HÌNH':^60}")
    print(f"• Alpha tối ưu của Lasso (BIC)      : {round(lasso_bic.alpha_, 5)}")
    print(f"• Chỉ số R2-score (Linear Regression) : {r2_score_lr}")
    print(f"• Chỉ số RMSE (Linear Regression)    : {rmse_lr}")
    print(f"• Chỉ số MAE (Linear Regression)     : {mae_lr}")
    print(f"• Số lượng tính năng bị loại bỏ      : {len(dropped_features)}/{len(feature_names)}")
    print(f"• Số lượng tính năng được giữ lại    : {len(kept_features)}/{len(feature_names)}")
    print("-"*60)
    if dropped_features:
        print(f"• Các tính năng bị loại bỏ (Lasso Coef = 0):\n  {dropped_features}\n")
    print(f"• Các tính năng được chọn để chạy Linear Regression:\n  {kept_features}")
    print("="*60)

NameError: name 'LassoLarsIC' is not defined

In [ ]:
import matplotlib.pyplot as plt

# 1. Lấy danh sách alphas và MSE trung bình tương ứng (LassoCV lưu MSE ở dạng số âm)
alphas = lasso.alphas_
mean_mse = np.mean(lasso.mse_path_, axis=1)

# 2. Vẽ biểu đồ
plt.figure(figsize=(8, 5))
plt.plot(alphas, mean_mse, label='MSE trung bình (5-Fold)', color='blue', lw=2)
plt.axvline(lasso.alpha_, color='red', linestyle='--', label=f'Alpha tối ưu ({round(lasso.alpha_, 5)})')

plt.xscale('log') # Vì alpha quét theo cấp số nhân
plt.xlabel('Hệ số Alpha (Log Scale)')
plt.ylabel('Sai số MSE')
plt.title('Biểu đồ tìm Alpha tối ưu bằng LassoCV')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

In [ ]:
#Các feature mà boruta và lasso cùng giữ lại
common_features = list(set(confirmed_features) & set(kept_features))
print(f"• Các tính năng được cả Boruta và Lasso giữ lại:\n  {common_features}")
boruta_only_features = list(set(confirmed_features) - set(kept_features))
print(f"• Các tính năng chỉ được Boruta giữ lại:\n  {boruta_only_features}")
lasso_only_features = list(set(kept_features) - set(confirmed_features))
print(f"• Các tính năng chỉ được Lasso giữ lại:\n  {lasso_only_features}")

Sau khi loại bỏ hoàn toàn các biến đối ngẫu gây ra hiện tượng đa cộng tuyến nghiêm trọng ($VIF = \infty$), một phát hiện thực nghiệm quan trọng đã được ghi nhận: tập các đặc trưng được chọn bởi Boruta tạo thành một tập con nghiêm ngặt của tập Lasso ($Boruta \subset Lasso$).Mặc dù Lasso sở hữu cơ chế tối giản hóa mô hình ($L_1$ penalty), thực nghiệm cho thấy Boruta lại có xu hướng nghiêm ngặt hơn khi lọc bỏ thêm 9 đặc trưng vùng biên (chủ yếu là các trạng thái phân loại của CAEC và MTRANS). Khi đưa hai tập tính năng này vào mô hình Hồi quy tuyến tính, chúng tôi quan sát thấy một sự đánh đổi về mặt thống kê: 9 đặc trưng thặng dư của Lasso giúp giảm nhẹ chỉ số RMSE (từ 5.335 xuống 5.327), cho thấy khả năng làm mượt các điểm ngoại lai. Tuy nhiên, tập 14 đặc trưng của Boruta lại mang lại chỉ số MAE tối ưu hơn (4.095 so với 4.119) và giữ nguyên mức giải thích vĩ mô $R^2 = 0.96$.Dựa trên nguyên lý tinh gọn (Parsimony), nghiên cứu này kết luận rằng quy trình chọn lọc của Boruta mang lại một kiến trúc dữ liệu hiệu quả hơn cho mô hình Hồi quy tuyến tính, giúp cắt giảm 39% số lượng biến (từ 23 xuống 14) mà không gây ra bất kỳ sự suy giảm có ý nghĩa nào về mặt hiệu năng tổng thể.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

def get_vif(features):
    X_subset = sm.add_constant(X_train[features])
    return pd.Series([variance_inflation_factor(X_subset.values, i) for i in range(X_subset.shape[1])],
                     index=X_subset.columns).drop('const')

print("VIF Boruta:")
display(get_vif(confirmed_features))

print("\nVIF Lasso:")
display(get_vif(lasso_only_features))